<!-- <img src="MUML_header.svg" alt="Header" style="width:100%; height:auto;"> -->
![MUML_header](MUML_header.svg)

# Mathematical Foundations. Automatic differentiation with PyTorch

In [ ]:
import torch

We have reviewed the theory of computing derivatives and the chain rule. Computing the derivative of a complex function by hand can be tedious and error-prone. We are going to explore one of PyTorch’s most powerful tools: its automatic differentiation engine, called autograd.

Let’s set neural networks aside for a moment. Think of PyTorch as a very advanced mathematical calculator. It can not only compute the value of a function, but it can also automatically compute the derivative of that function with respect to any of its input variables.

All modern deep learning frameworks offer automatic differentiation (often shortened to autograd). As we pass data through each successive function, the framework builds a computational graph that tracks how each value depends on others. To calculate derivatives, automatic differentiation works backwards through this graph applying the chain rule. The computational algorithm for applying the chain rule in this fashion is called backpropagation.

To illustrate how it works, we will work with the following multivariable function, evaluated at points: `a=3`, `b=4`, `c=5`.

$$f(a,b,c)=(a*b+c)*2$$

Our goal is to use PyTorch to compute the partial derivatives of $f$ with respect to each input:
* $\frac{\partial f}{\partial a}$ - how $f$ changes when we vary $a$
* $\frac{\partial f}{\partial b}$ - how $f$ changes when we vary $b$
* $\frac{\partial f}{\partial c}$ - how $f$ changes when we vary $c$

We start by creating tensors for our input variables a, b, and c.

For any variable we want to compute derivatives with respect to, we must specify `requires_grad=True`. This tells PyTorch to keep track of every operation performed on the tensor so gradients can be calculated later.

At this stage, the `.grad` attribute is None because no gradients have been computed yet.

In [ ]:
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)
c = torch.tensor(5.0, requires_grad=True)

print(f"Initial state:")
print(f"a = {a}, a.grad = {a.grad}")
print(f"b = {b}, b.grad = {b.grad}")
print(f"c = {c}, c.grad = {c.grad}\n")

We write the function, using the variables already created.

When we write this function in code, PyTorch is silently building a computational graph that records the sequence of operations.

In [ ]:
print("--- Computing the function output ---")

f = (a * b + c) * 2   

print(f"Final result f: {f}\n")


Calling `.backward()` on the final tensor `f` tells PyTorch to traverse the computational graph in reverse and apply the chain rule to compute the gradients of `f` with respect to all tensors where `requires_grad=True`.

In [ ]:
print("--- Running f.backward() ---")
f.backward()
print(".backward() executed!\n")

PyTorch does not directly return the gradient values. Instead, it stores them in the `.grad` attribute of each input tensor.

In [ ]:
print("Final State (with gradients calculated):")
print(f"a = {a}, a.grad (∂f/∂a) = {a.grad}")
print(f"b = {b}, b.grad (∂f/∂b) = {b.grad}")
print(f"c = {c}, c.grad (∂f/∂c) = {c.grad}\n")

Finally, let’s verify that the results are correct by hand using the chain rule.

Recall:

$$f(a,b,c)=(a*b+c)*2$$

* $\frac{\partial f}{\partial a}=2b=2*4.0=8.0$ 
* $\frac{\partial f}{\partial b}=2a=2*3.0=6.0$ 
* $\frac{\partial f}{\partial c}=2.0$ 

In [ ]:
print("--- Manual Verification ---")
print(f"∂f/∂a computed manually: 8.0. PyTorch value (a.grad): {a.grad}")
print(f"∂f/∂b computed manually: 6.0. PyTorch value (b.grad): {b.grad}")
print(f"∂f/∂c computed manually: 2.0. PyTorch value (c.grad): {c.grad}")

When you perform operations on tensors that have `requires_grad=True`, PyTorch dynamically builds a computational graph. This graph keeps track of how each tensor was created so that gradients can later be computed automatically.

* Nodes in this graph represent tensors.
* Edges represent the operations that produced those tensors.

For example, in our example function $f(a,b,c)=(a*b+c)*2$, PyTorch internally recorded that `f` depends on `e = a*b + c`, which in turn depends on `d = a*b`, and so on.

We can actually inspect this graph. Every tensor that is the result of an operation has an attribute called `grad_fn`, which tells us the function (operation) that created it. From there, the attribute `.next_functions` allows us to see what came before — essentially, the parent nodes in the graph. By following these links, you can trace the graph step by step back to the original inputs (`a`, `b`, `c`).

Finally, when you called `f.backward()`, PyTorch started at the output tensor `f` and traversed the graph in reverse order, applying the chain rule at each step. This is how it automatically computed the derivatives of `f` with respect to `a`, `b`, and `c`.

In [ ]:
last_function=f.grad_fn
print(f"The tensor 'f' was created by the function: {last_function}")
print(f"The functions operands for that function were: {last_function.next_functions}")
print(f"The functions operands for that function were: {last_function.next_functions[0][0].next_functions}")
print(f"The functions operands for that function were: {last_function.next_functions[0][0].next_functions[0][0].next_functions}")

~~~
   a      b       c
    \    /        |
     Mul (a*b)    |
       |          |
       d ---------+
          Add (d+c)
              |
              e
              |
          Mul (*2)
              |
              f


Just for demonstration purposes, we repeat the same code with intermediate variables and enable them to store their gradients using `.retain_grad()`. Keep in mind that this is not common practice, since it increases memory usage. Normally, only leaf tensors (like `a`, `b`, and `c`) store their gradients in `.grad`. Intermediate variables do not retain gradients by default, allowing PyTorch to free memory and optimize the computation.ckward0)

In [ ]:
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)
c = torch.tensor(5.0, requires_grad=True)
d=a*b
e=d+c
f=2*e
d.retain_grad()
e.retain_grad()
f.retain_grad()
f.backward()
#Print the gradients of all variables
print("a.grad =", a.grad)  # ∂f/∂a = 8
print("b.grad =", b.grad)  # ∂f/∂b = 6
print("c.grad =", c.grad)  # ∂f/∂c = 2
print("d.grad =", d.grad)  # ∂f/∂d = 2
print("e.grad =", e.grad)  # ∂f/∂e = 2
print("f.grad =", f.grad)  # ∂f/∂f = 1

In PyTorch, gradients are accumulated by default in the `.grad` attribute of each tensor. This design choice is intentional: it allows you to perform gradient accumulation over multiple mini-batches before updating your model parameters (useful, for example, when the batch size is too large to fit in memory).

However, in most training loops you only want the gradient values from the current backward pass. If you don’t reset them, the new gradients will be added to the existing ones, leading to incorrect updates.

That’s why, before calling `.backward()` again, you typically clear the gradients with the `.zero_grad()` method (working with optimizers) or, working directly with tensors, with `.grad.zero_()`.

In [ ]:
#Without changing any value, if we compute the function again, the gradients are added
d=a*b
e=d+c
f=2*e
f.backward()
#Print the gradients of all variables
print("a.grad =", a.grad)  # ∂f/∂a = 16
print("b.grad =", b.grad)  # ∂f/∂b = 12
print("c.grad =", c.grad)  # ∂f/∂c = 4

In [ ]:
#Reset the gradients to zero before computing
a.grad.zero_()
b.grad.zero_()
c.grad.zero_()
d=a*b
e=d+c
f=2*e
f.backward()
#Print the gradients of all variables
print("a.grad =", a.grad)  # ∂f/∂a = 8
print("b.grad =", b.grad)  # ∂f/∂b = 6
print("c.grad =", c.grad)  # ∂f/∂c = 2

Sometimes, you don't want PyTorch to track operations for gradient computation. Comon examples are:
* When you have a trained model and just want to make predictions
* when updating parameters (directly modify tensor values)
* For extra operations, like computing metrics or statistics

In those cases, you can specify `torch.no_grad()`. If you forget to do it during inference (for example), your code will still work, but it will consume more memory and be less efficient, since PyTorch will unnecessarily build the gradient graph.

In [ ]:
with torch.no_grad():
    a=a+2
    b=b+2
    c=c+2

So far we reviewed cases where the path from input to output was well defined via a function such as `z = x * x * x`. Programming offers us a lot more freedom in how we compute results. For instance, we can make them depend on auxiliary variables or condition choices on intermediate results. One benefit of using automatic differentiation is that even if building the computational graph of a function required passing through a maze of Python control flow (e.g., conditionals, loops, and arbitrary function calls), we can still calculate the gradient of the resulting variable. 

To illustrate this, consider the following code snippet where the number of iterations of the while loop and the evaluation of the if statement both depend on the value of the input a.

In [ ]:
def f(a):
    b = a * 2
    while b.abs() < 100:
        b = b * 5
    if b > 0:
        c = b/10
    else:
        c = -10 * b
    return c

Below, we call this function, passing in a random value, as input. Since the input is a random variable, we do not know what form the computational graph will take. However, whenever we execute `f(a)` on a specific input, we realize a specific computational graph and can subsequently run backward.

In [ ]:
a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()

print(f"The tensor 'f' was created by the function: {d.grad_fn}")

# Mathematical Foundations. Automatic differentiation with TensorFlow

During this subject we will use Pytorch. However, this is not the only framework used for deep learning. Another common one is TensorFlow. We are going to repeat the same execution as before using TensorFlow. 

Just as PyTorch provides its `autograd` engine, TensorFlow also includes a powerful automatic differentiation system. In TensorFlow, this functionality is centered around the `tf.GradientTape` context manager, which records operations performed on tensors so that gradients can be computed later.

The idea is the same: TensorFlow builds a computational graph as you apply operations to tensors. This graph captures how outputs depend on inputs. When it comes time to compute derivatives, TensorFlow traverses the graph in reverse, applying the chain rule step by step — the process commonly known as backpropagation.

Let’s revisit the same multivariable function we used before, evaluated at the points `a = 3`, `b = 4`, `c = 5`:

$$f(a,b,c)=(a*b+c)*2$$

Our goal is again to compute the partial derivatives of $f$ with respect to each input variable:
* $\frac{\partial f}{\partial a}$ - how $f$ changes when we vary $a$
* $\frac{\partial f}{\partial b}$ - how $f$ changes when we vary $b$
* $\frac{\partial f}{\partial c}$ - how $f$ changes when we vary $c$

The key difference from PyTorch is that in TensorFlow we explicitly wrap the computation inside a `GradientTape`. The tape records every operation performed on “watched” tensors (in our case, a, b, and c), and then we can ask it for the gradients of the output with respect to these inputs.

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf

We start by creating tensors for our input variables `a`, `b`, and `c`. In TensorFlow, any parameter that we intend to train or compute gradients with respect to should be defined as a `tf.Variable`. This is a special object that signals to TensorFlow that its state can be modified and that operations involving it should be tracked.

In [ ]:
# We use tf.Variable for any parameter we need to compute gradients for.
a = tf.Variable(3.0)
b = tf.Variable(4.0)
c = tf.Variable(5.0)

print(f"Initial state:")
# We use .numpy() to see the clean numerical value of the tensor.
print(f"a = {a.numpy()}")
print(f"b = {b.numpy()}")
print(f"c = {c.numpy()}\n")

To compute gradients, TensorFlow uses a context manager called `tf.GradientTape`. Think of it as a recording device: any calculation involving a `tf.Variable` that happens inside a with `tf.GradientTape() as tape:` block will be recorded onto the "tape". This recording is essentially the computational graph for that specific forward pass.

In [ ]:
print("--- Computing the function output ---")

# We open a GradientTape to record the operations.
with tf.GradientTape() as tape:
    # We define our function *inside* the tape's context.
    f = (a * b + c) * 2

print(f"Final result f: {f.numpy()}\n")

After the forward pass is complete, we use the `tape` object to compute the gradients. We call `tape.gradient()`, passing it the final result (`f`, the "target") and the variables we want to differentiate with respect to (`a`, `b`, and `c`, the "sources").

In [ ]:
print("--- Calculating gradients using tape.gradient() ---")

# The tape computes the gradient of the target (f) with respect to the sources.
# It returns a list of gradient tensors, one for each source.
gradients = tape.gradient(f, [a, b, c])

print("Gradients calculated!\n")

The `tape.gradient()` call returns a list of gradient tensors corresponding to the list of source variables provided. We can now inspect these results.

In [ ]:
# The returned gradients are in the same order as the input variables.
grad_a = gradients[0]
grad_b = gradients[1]
grad_c = gradients[2]

print("Calculated Gradients:")
print(f"∂f/∂a = {grad_a.numpy()}")
print(f"∂f/∂b = {grad_b.numpy()}")
print(f"∂f/∂c = {grad_c.numpy()}\n")

Finally, let’s verify that the results are correct by hand using the chain rule. Recall:
$$f(a,b,c)=(a*b+c)*2$$

* $\frac{\partial f}{\partial a}=2b=2*4.0=8.0$ 
* $\frac{\partial f}{\partial b}=2a=2*3.0=6.0$ 
* $\frac{\partial f}{\partial c}=2.0$ 2.0

In [ ]:
print("--- Manual Verification ---")
print(f"∂f/∂a computed manually: 8.0. TensorFlow value: {grad_a.numpy()}")
print(f"∂f/∂b computed manually: 6.0. TensorFlow value: {grad_b.numpy()}")
print(f"∂f/∂c computed manually: 2.0. TensorFlow value: {grad_c.numpy()}")

When you performed the calculations inside the `tf.GradientTape` block, TensorFlow built an internal graph representing the sequence of operations. The `tape` object holds this record. Unlike in PyTorch, you don't typically inspect this graph by traversing `.grad_fn` attributes in TensorFlow's default (Eager) execution mode. Instead, the GradientTape itself is the primary interface for interacting with the recorded computation. When you call `tape.gradient()`, TensorFlow uses its record of the forward pass to traverse the graph in reverse and apply the chain rule, just like in PyTorch.


A powerful feature of GradientTape is that it can compute gradients with respect to any tensor that was computed inside its context, not just the initia`l tf.Variabl`es. This is the idiomatic way to inspect intermediate gradients in TensorFlow.

In [ ]:
a = tf.Variable(3.0)
b = tf.Variable(4.0)
c = tf.Variable(5.0)

with tf.GradientTape() as tape:
    # We create the same intermediate variables
    d = a * b
    e = d + c
    f = 2 * e

# We can ask for the gradient with respect to all variables, including intermediates.
all_gradients = tape.gradient(f, [a, b, c, d, e, f])

print("--- Gradients for all variables ---")
print("a.grad =", all_gradients[0].numpy()) # ∂f/∂a = 8
print("b.grad =", all_gradients[1].numpy()) # ∂f/∂b = 6
print("c.grad =", all_gradients[2].numpy()) # ∂f/∂c = 2
print("d.grad =", all_gradients[3].numpy()) # ∂f/∂d = 2
print("e.grad =", all_gradients[4].numpy()) # ∂f/∂e = 2
print("f.grad =", all_gradients[5].numpy()) # ∂f/∂f = 1

A fundamental difference in TensorFlow's design is that gradients are NOT accumulated by default. The `GradientTape` is an ephemeral object. Once you call `tape.gradient()`, its resources are typically released. For the next step in an optimization loop, you create a new tape.

This means you do not need to reset gradients manually. This design avoids the common error of forgetting to zero out gradients.

In [ ]:
a = tf.Variable(3.0)
b = tf.Variable(4.0)
c = tf.Variable(5.0)

# First calculation
with tf.GradientTape() as tape1:
    f1 = (a * b + c) * 2
gradients1 = tape1.gradient(f1, [a, b, c])
print("First calculation, a.grad =", gradients1[0].numpy()) # ∂f/∂a = 8

# Second calculation
# We create a completely new tape. The old one is gone.
with tf.GradientTape() as tape2:
    f2 = (a * b + c) * 2
gradients2 = tape2.gradient(f2, [a, b, c])

# The gradient is calculated fresh and is not added to the previous one.
print("Second calculation, a.grad =", gradients2[0].numpy()) # ∂f/∂a = 8

### Summary: Key Differences Between PyTorch and TensorFlow for autograd While both frameworks achieve the same goal, their approach and syntax are different.


* Trainable variables:
    * PyTorch: any tensor can be marked for tracking using `requires_grad=True`.
    * Tensorflow: There is a special object (`tf.Variable`) for things that are meant to be trained. Non trainable data is a `tf.constant`.

* Gradient calculation:
    * PyTorch: the `.backward()`call is a method on the loss tensor itself. It computes and stores gradients.
    * TensorFlow: You must explicitly define a context (GradientTape) where operations are recorded. The tape object is then use to compute the gradients.

* Accessing gradients:
    * PyTorch: Gradients are stored in the `.grad` attribute of the tensor.
    * TensorFlow: The `tape.gradient()` function returns a list of gradient tensors.
 
* Updating variables:
    * PyTorch: You use standard Python operators (-=) and must wrap the update in a no_grad() block to prevent it from being tracked.
    * TensorFlow: tf.Variable has special methods like .assign() and .assign_sub() for efficient, in-place updates. Operations outside the tape are not tracked by default anyway.
 
* Updating variables:
    * PyTorch: `.backward()` accumulates gradients into the `.grad` attribute. You must clear them to prevent mixing gradients from different steps.
    * TensorFlow: The `GradientTape` is ephemeral. It's created, used once, and discarded. Gradients are recalculated from scratch in each loop, so there's nothing to accumulate or reset. reset.